# Experiment 08 — External Paper Reproduction

## E7 — Linear Baseline Reproduction

### 1. Mục tiêu

Tái thực nghiệm mô hình **LTSF-Linear** được giới thiệu trong bài báo *Are Transformers Effective for Time Series Forecasting?* (Zeng et al., AAAI 2023).

Mục tiêu:
- Triển khai kiến trúc Linear theo mã nguồn gốc.
- Huấn luyện và đánh giá trên ETTh1.
- Thiết lập baseline để kiểm tra ảnh hưởng của thứ tự thời gian trong các thí nghiệm tiếp theo.

### 2. Thiết kế thực nghiệm

**Dataset:** ETTh1 — 7 channels.

**Model:** LTSF-Linear.

**Task:** Multivariate-to-Multivariate Forecasting.

**Forecasting Horizons:** 96, 192, 336, 720.

**Input Sequence Length:** 96 — cấu hình đang sử dụng cho thí nghiệm Temporal Order.

**Data Preparation:**
- Chronological Train/Validation/Test Split.
- StandardScaler fitted on Train only.
- Sliding Window Generation.

### 3. Kiến trúc mô hình

LTSF-Linear sử dụng một lớp tuyến tính để ánh xạ trực tiếp chuỗi quá khứ thành chuỗi dự báo tương lai.

\[
\hat{X}=WX+b
\]

Ánh xạ được thực hiện theo chiều thời gian, với trọng số dùng chung giữa các channels.

### 4. Quy trình thực hiện

1. Nạp dữ liệu ETTh1 đã tiền xử lý.
2. Khởi tạo mô hình Linear.
3. Huấn luyện trên Train và theo dõi Validation.
4. Đánh giá trên Test.
5. Lặp lại với các forecasting horizons và random seeds được thiết lập.
6. Lưu checkpoint, MSE, MAE và thời gian huấn luyện.

### 5. Kết quả đầu ra

- Checkpoints của mô hình Linear.
- `metrics.csv` — Kết quả Train/Validation/Test.
- Bảng tổng hợp MSE và MAE.
- Dữ liệu phục vụ thí nghiệm Temporal Order Sensitivity.

Kết quả sẽ được đối chiếu với bài báo gốc và sử dụng làm baseline cho DeepKoopFormer.

LINEAR REPRODUCTION: ENVIRONMENT & DATA

In [1]:
from pathlib import Path
import hashlib
import random

import numpy as np
import pandas as pd
import torch

# 1. Reproducibility
SEED = 7
EVAL_SEEDS = [7, 17, 27, 37, 47]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 2. Locate project root
PROJECT_ROOT = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (
            p / "datasets" / "ETTh1" / "ETTh1.csv"
        ).is_file()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate datasets/ETTh1/ETTh1.csv"
    )

# 3. Directories
EXTERNAL_DIR = (
    PROJECT_ROOT / "experiments" / "08_external_paper"
)

RAW_DATA_PATH = (
    PROJECT_ROOT / "datasets" / "ETTh1" / "ETTh1.csv"
)

PREPARED_PATH = (
    EXTERNAL_DIR / "results"
    / "etth1_data" / "etth1_prepared.pt"
)

RESULTS_DIR = (
    EXTERNAL_DIR / "results" / "linear_reproduction"
)

MODEL_DIR = RESULTS_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

assert PREPARED_PATH.is_file(), (
    "Run 01_etth1_data.ipynb first"
)

# 4. Load prepared dataset
bundle = torch.load(
    PREPARED_PATH,
    map_location="cpu",
    weights_only=True
)

assert bundle["dataset"] == "ETTh1"
assert bundle["normalization"] == "StandardScaler"

# Verify original dataset fingerprint
raw_sha256 = hashlib.sha256(
    RAW_DATA_PATH.read_bytes()
).hexdigest()

assert bundle["source_sha256"] == raw_sha256, (
    "Raw ETTh1 file does not match prepared dataset"
)

# 5. Restore configuration
SEQ_LEN = int(bundle["seq_len"])
PRED_LENGTHS = list(bundle["pred_lengths"])
N_CHANNELS = int(bundle["channels"])
FEATURE_COLUMNS = list(bundle["features"])

SCALER_MEAN = bundle["scaler_mean"]
SCALER_SCALE = bundle["scaler_scale"]

SPLIT_BORDERS = bundle["split_borders"]
TARGET_STARTS = bundle["target_starts"]

ETTH1_SPLITS = {
    split: bundle[split].float()
    for split in ["train", "val", "test"]
}

# 6. Validate shapes and configuration
assert SEQ_LEN == 96
assert PRED_LENGTHS == [96, 192, 336, 720]
assert N_CHANNELS == 7

EXPECTED_SHAPES = {
    "train": (8640, 7),
    "val": (2976, 7),
    "test": (2976, 7)
}

for split, data in ETTH1_SPLITS.items():
    assert tuple(data.shape) == EXPECTED_SHAPES[split]
    assert torch.isfinite(data).all()

assert SCALER_MEAN.shape == (N_CHANNELS,)
assert SCALER_SCALE.shape == (N_CHANNELS,)
assert (SCALER_SCALE > 0).all()

# 7. Validate historical context overlap
# Validation uses the last 96 Train timestamps as input.
assert torch.equal(
    ETTH1_SPLITS["val"][:SEQ_LEN],
    ETTH1_SPLITS["train"][-SEQ_LEN:]
)

# Test uses the last 96 Validation timestamps as input.
assert torch.equal(
    ETTH1_SPLITS["test"][:SEQ_LEN],
    ETTH1_SPLITS["val"][-SEQ_LEN:]
)

# 8. Validate Train standardization
train = ETTH1_SPLITS["train"]

np.testing.assert_allclose(
    train.mean(dim=0).numpy(),
    np.zeros(N_CHANNELS),
    atol=1e-4
)

np.testing.assert_allclose(
    train.std(dim=0, unbiased=False).numpy(),
    np.ones(N_CHANNELS),
    atol=1e-4
)

# 9. Dataset summary
rows = []

for split, data in ETTH1_SPLITS.items():
    row = {
        "Split": split,
        "Shape": str(tuple(data.shape))
    }

    for horizon in PRED_LENGTHS:
        row[f"Windows_H{horizon}"] = (
            len(data) - SEQ_LEN - horizon + 1
        )

    rows.append(row)

data_summary = pd.DataFrame(rows)

# 10. Display
print("=== E7 — LINEAR REPRODUCTION SETUP ===")

print("Project:", PROJECT_ROOT)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nSource:", PREPARED_PATH)
print("Source SHA256 verified")

print("\nDataset: ETTh1")
print("Features:", FEATURE_COLUMNS)
print("Channels:", N_CHANNELS)

print("\nSequence length:", SEQ_LEN)
print("Forecast horizons:", PRED_LENGTHS)
print("Experiment seeds:", EVAL_SEEDS)

print("\n=== DATA SPLITS ===")
display(data_summary)

print("\nNormalization: StandardScaler (Train only)")
print("Historical context boundaries: VERIFIED")

print("\nResults directory:", RESULTS_DIR)
print("Models directory:", MODEL_DIR)

print("\nTraining performed: NO")
print("LINEAR REPRODUCTION SETUP PASSED!")


=== E7 — LINEAR REPRODUCTION SETUP ===
Project: c:\Users\Win 11 New\Desktop\pa gk
Device: cuda
GPU: Quadro T1000

Source: c:\Users\Win 11 New\Desktop\pa gk\experiments\08_external_paper\results\etth1_data\etth1_prepared.pt
Source SHA256 verified

Dataset: ETTh1
Features: ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']
Channels: 7

Sequence length: 96
Forecast horizons: [96, 192, 336, 720]
Experiment seeds: [7, 17, 27, 37, 47]

=== DATA SPLITS ===


c:\Users\Win 11 New\Desktop\pa gk\.venv\lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


,Split,Shape,Windows_H96,Windows_H192,Windows_H336,Windows_H720
0,train,"(8640, 7)",8449,8353,8209,7825
1,val,"(2976, 7)",2785,2689,2545,2161
2,test,"(2976, 7)",2785,2689,2545,2161



Normalization: StandardScaler (Train only)
Historical context boundaries: VERIFIED

Results directory: c:\Users\Win 11 New\Desktop\pa gk\experiments\08_external_paper\results\linear_reproduction
Models directory: c:\Users\Win 11 New\Desktop\pa gk\experiments\08_external_paper\results\linear_reproduction\models

Training performed: NO
LINEAR REPRODUCTION SETUP PASSED!


LTSF-LINEAR ARCHITECTURE

In [2]:
import torch.nn as nn

# 1. Architecture based on official LTSF-Linear
class LTSFLinear(nn.Module):

    def __init__(
        self,
        seq_len,
        pred_len,
        channels,
        individual=False
    ):
        super().__init__()

        self.seq_len = int(seq_len)
        self.pred_len = int(pred_len)
        self.channels = int(channels)
        self.individual = bool(individual)

        if self.individual:
            self.Linear = nn.ModuleList([
                nn.Linear(self.seq_len, self.pred_len)
                for _ in range(self.channels)
            ])
        else:
            self.Linear = nn.Linear(
                self.seq_len,
                self.pred_len
            )

    def forward(self, x):
        # x: [Batch, Sequence Length, Channels]

        assert x.ndim == 3
        assert x.shape[1] == self.seq_len
        assert x.shape[2] == self.channels

        if self.individual:
            # Separate Linear layer for each channel
            output = torch.stack(
                [
                    self.Linear[c](x[:, :, c])
                    for c in range(self.channels)
                ],
                dim=-1
            )
        else:
            # Shared Linear layer across channels
            output = self.Linear(
                x.permute(0, 2, 1)
            ).permute(0, 2, 1)

        return output


# 2. Reproduction configuration
INDIVIDUAL = False

LINEAR_CONFIG = {
    "seq_len": SEQ_LEN,
    "channels": N_CHANNELS,
    "individual": INDIVIDUAL
}

# 3. Instantiate and validate all four horizons
MODEL_SPECS = {}
rows = []

for horizon in PRED_LENGTHS:

    torch.manual_seed(SEED)

    model = LTSFLinear(
        seq_len=SEQ_LEN,
        pred_len=horizon,
        channels=N_CHANNELS,
        individual=INDIVIDUAL
    ).to(DEVICE)

    model.eval()

    # Use actual ETTh1 input for forward-pass test
    X_dummy = (
        ETTH1_SPLITS["train"][:SEQ_LEN]
        .unsqueeze(0)
        .to(DEVICE)
    )

    with torch.inference_mode():
        Y_dummy = model(X_dummy)

    expected_shape = (1, horizon, N_CHANNELS)

    assert tuple(Y_dummy.shape) == expected_shape
    assert torch.isfinite(Y_dummy).all()

    # Parameter count for shared Linear
    expected_params = (
        (SEQ_LEN * horizon + horizon)
        if not INDIVIDUAL
        else N_CHANNELS * (SEQ_LEN * horizon + horizon)
    )

    actual_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    assert actual_params == expected_params

    # Output of a given channel must not depend
    # on values in other channels
    X_modified = X_dummy.clone()
    X_modified[:, :, 1] += 10.0

    with torch.inference_mode():
        Y_modified = model(X_modified)

    torch.testing.assert_close(
        Y_dummy[:, :, 0],
        Y_modified[:, :, 0],
        rtol=0,
        atol=0
    )

    MODEL_SPECS[horizon] = {
        "seq_len": SEQ_LEN,
        "pred_len": horizon,
        "channels": N_CHANNELS,
        "individual": INDIVIDUAL,
        "parameters": actual_params
    }

    rows.append({
        "Horizon": horizon,
        "Input Shape": str(tuple(X_dummy.shape)),
        "Output Shape": str(tuple(Y_dummy.shape)),
        "Parameters": actual_params,
        "Shared Weights": not INDIVIDUAL,
        "Status": "PASSED"
    })

    del model


# 4. Architecture summary
ARCHITECTURE_DF = pd.DataFrame(rows)

assert len(ARCHITECTURE_DF) == 4
assert (ARCHITECTURE_DF["Status"] == "PASSED").all()

print("=== LTSF-LINEAR ARCHITECTURE ===")
print("Model: Linear")
print("Input length:", SEQ_LEN)
print("Channels:", N_CHANNELS)
print("Individual:", INDIVIDUAL)
print("Weight sharing:", "YES" if not INDIVIDUAL else "NO")

display(ARCHITECTURE_DF)

print("\nTotal architectures verified:", len(MODEL_SPECS))
print("Training performed: NO")
print("\nLTSF-LINEAR ARCHITECTURE PASSED!")


=== LTSF-LINEAR ARCHITECTURE ===
Model: Linear
Input length: 96
Channels: 7
Individual: False
Weight sharing: YES


,Horizon,Input Shape,Output Shape,Parameters,Shared Weights,Status
0,96,"(1, 96, 7)","(1, 96, 7)",9312,True,PASSED
1,192,"(1, 96, 7)","(1, 192, 7)",18624,True,PASSED
2,336,"(1, 96, 7)","(1, 336, 7)",32592,True,PASSED
3,720,"(1, 96, 7)","(1, 720, 7)",69840,True,PASSED



Total architectures verified: 4
Training performed: NO

LTSF-LINEAR ARCHITECTURE PASSED!


TRAINING CONFIGURATION & DATALOADERS

In [3]:
import json
from torch.utils.data import Dataset, DataLoader

# 1. Training configuration
TRAIN_CONFIG = {
    "epochs": 10,
    "batch_size": 32,
    "learning_rate": 0.005,
    "patience": 3,
    "optimizer": "Adam",
    "loss": "MSE",
    "lr_schedule": "type1",
    "individual": False,
    "num_workers": 0
}

# Note: LR=0.005 is adapted from the official
# ETTh1 DLinear script, not independently
# verified as Table 5 Linear's exact setting.

# 2. Sliding-window Dataset
class LinearWindowDataset(Dataset):

    def __init__(self, series, seq_len, pred_len):
        self.series = series
        self.seq_len = int(seq_len)
        self.pred_len = int(pred_len)

        assert series.ndim == 2
        assert series.shape[1] == N_CHANNELS
        assert len(series) >= seq_len + pred_len

    def __len__(self):
        return (
            len(self.series)
            - self.seq_len
            - self.pred_len
            + 1
        )

    def __getitem__(self, idx):
        if idx < 0 or idx >= len(self):
            raise IndexError(idx)

        x_start = idx
        x_end = idx + self.seq_len
        y_end = x_end + self.pred_len

        X = self.series[x_start:x_end]
        Y = self.series[x_end:y_end]

        return X, Y


# 3. DataLoader factory
# Create loaders separately for each run/seed.
def make_dataloaders(horizon, seed):

    assert horizon in PRED_LENGTHS

    datasets = {
        split: LinearWindowDataset(
            series=ETTH1_SPLITS[split],
            seq_len=SEQ_LEN,
            pred_len=horizon
        )
        for split in ["train", "val", "test"]
    }

    generator = torch.Generator()
    generator.manual_seed(int(seed))

    common = {
        "batch_size": TRAIN_CONFIG["batch_size"],
        "num_workers": TRAIN_CONFIG["num_workers"],
        "pin_memory": DEVICE.type == "cuda"
    }

    loaders = {
        "train": DataLoader(
            datasets["train"],
            shuffle=True,
            drop_last=True,
            generator=generator,
            **common
        ),
        "val": DataLoader(
            datasets["val"],
            shuffle=False,
            drop_last=False,
            **common
        ),
        "test": DataLoader(
            datasets["test"],
            shuffle=False,
            drop_last=False,
            **common
        )
    }

    return datasets, loaders


# 4. Validate all forecasting horizons
loader_rows = []

for horizon in PRED_LENGTHS:

    datasets, loaders = make_dataloaders(
        horizon=horizon,
        seed=SEED
    )

    for split in ["train", "val", "test"]:

        ds = datasets[split]
        loader = loaders[split]

        X_batch, Y_batch = next(iter(loader))

        assert X_batch.ndim == 3
        assert Y_batch.ndim == 3

        assert X_batch.shape[1:] == (
            SEQ_LEN, N_CHANNELS
        )

        assert Y_batch.shape[1:] == (
            horizon, N_CHANNELS
        )

        assert torch.isfinite(X_batch).all()
        assert torch.isfinite(Y_batch).all()

        # Verify target follows input immediately.
        X_first, Y_first = ds[0]

        assert torch.equal(
            X_first,
            ETTH1_SPLITS[split][:SEQ_LEN]
        )

        assert torch.equal(
            Y_first,
            ETTH1_SPLITS[split][
                SEQ_LEN:SEQ_LEN + horizon
            ]
        )

        loader_rows.append({
            "Horizon": horizon,
            "Split": split,
            "Windows": len(ds),
            "Batches": len(loader),
            "X_Batch_Shape": str(tuple(X_batch.shape)),
            "Y_Batch_Shape": str(tuple(Y_batch.shape)),
            "Status": "PASSED"
        })


# 5. Validate deterministic Train shuffling
_, loaders_a = make_dataloaders(96, SEED)
_, loaders_b = make_dataloaders(96, SEED)

xa, ya = next(iter(loaders_a["train"]))
xb, yb = next(iter(loaders_b["train"]))

assert torch.equal(xa, xb)
assert torch.equal(ya, yb)

# 6. Save training protocol
protocol = {
    "dataset": "ETTh1",
    "model": "LTSF-Linear",
    "task": "multivariate_to_multivariate",
    "seq_len": SEQ_LEN,
    "pred_lengths": PRED_LENGTHS,
    "channels": N_CHANNELS,
    "seeds": EVAL_SEEDS,
    "training": TRAIN_CONFIG,
    "source_sha256": raw_sha256,
    "protocol_note": (
        "Linear, seq_len=96. LR=0.005 adapted "
        "from official ETTh1 DLinear script. "
        "Validation uses all windows."
    )
}

PROTOCOL_PATH = RESULTS_DIR / "training_protocol.json"

with open(PROTOCOL_PATH, "w", encoding="utf-8") as f:
    json.dump(protocol, f, indent=2)

# 7. Summary
LOADER_SUMMARY = pd.DataFrame(loader_rows)

assert len(LOADER_SUMMARY) == 12
assert (LOADER_SUMMARY["Status"] == "PASSED").all()

print("=== LTSF-LINEAR TRAINING CONFIGURATION ===")

for key, value in TRAIN_CONFIG.items():
    print(f"{key}: {value}")

print("\nRandom seeds:", EVAL_SEEDS)
print("Forecast horizons:", PRED_LENGTHS)
print("Planned training runs:",
      len(PRED_LENGTHS) * len(EVAL_SEEDS))

print("\n=== DATALOADER VALIDATION ===")
display(LOADER_SUMMARY)

print("\nTrain shuffling reproducibility: VERIFIED")
print("Saved:", PROTOCOL_PATH)
print("Training performed: NO")

print("\nTRAINING CONFIGURATION PASSED!")


=== LTSF-LINEAR TRAINING CONFIGURATION ===
epochs: 10
batch_size: 32
learning_rate: 0.005
patience: 3
optimizer: Adam
loss: MSE
lr_schedule: type1
individual: False
num_workers: 0

Random seeds: [7, 17, 27, 37, 47]
Forecast horizons: [96, 192, 336, 720]
Planned training runs: 20

=== DATALOADER VALIDATION ===


,Horizon,Split,Windows,Batches,X_Batch_Shape,Y_Batch_Shape,Status
0,96,train,8449,264,"(32, 96, 7)","(32, 96, 7)",PASSED
1,96,val,2785,88,"(32, 96, 7)","(32, 96, 7)",PASSED
2,96,test,2785,88,"(32, 96, 7)","(32, 96, 7)",PASSED
3,192,train,8353,261,"(32, 96, 7)","(32, 192, 7)",PASSED
4,192,val,2689,85,"(32, 96, 7)","(32, 192, 7)",PASSED
5,192,test,2689,85,"(32, 96, 7)","(32, 192, 7)",PASSED
6,336,train,8209,256,"(32, 96, 7)","(32, 336, 7)",PASSED
7,336,val,2545,80,"(32, 96, 7)","(32, 336, 7)",PASSED
8,336,test,2545,80,"(32, 96, 7)","(32, 336, 7)",PASSED
9,720,train,7825,244,"(32, 96, 7)","(32, 720, 7)",PASSED



Train shuffling reproducibility: VERIFIED
Saved: c:\Users\Win 11 New\Desktop\pa gk\experiments\08_external_paper\results\linear_reproduction\training_protocol.json
Training performed: NO

TRAINING CONFIGURATION PASSED!


LTSF-LINEAR TRAINING & CHECKPOINTING

In [4]:
import copy
import time
import torch.nn.functional as F

CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"
HISTORY_DIR = RESULTS_DIR / "history"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_DIR.mkdir(parents=True, exist_ok=True)

METRICS_PATH = RESULTS_DIR / "metrics.csv"

# 1. Atomic checkpoint saving
def atomic_save(obj, path):
    temp = path.with_name(path.name + ".tmp")
    torch.save(obj, temp)
    temp.replace(path)


# 2. Evaluation on standardized and original scales
@torch.inference_mode()
def evaluate_linear(model, loader):

    model.eval()

    scale = SCALER_SCALE.to(
        device=DEVICE, dtype=torch.float32
    ).reshape(1, 1, -1)

    total_count = 0
    sum_sq = 0.0
    sum_abs = 0.0
    sum_sq_raw = 0.0
    sum_abs_raw = 0.0

    for X, Y in loader:
        X = X.to(DEVICE, non_blocking=True)
        Y = Y.to(DEVICE, non_blocking=True)

        pred = model(X)
        diff = pred - Y
        raw_diff = diff * scale

        total_count += diff.numel()

        sum_sq += diff.square().sum().item()
        sum_abs += diff.abs().sum().item()

        sum_sq_raw += raw_diff.square().sum().item()
        sum_abs_raw += raw_diff.abs().sum().item()

    assert total_count > 0

    return {
        "MSE": sum_sq / total_count,
        "MAE": sum_abs / total_count,
        "MSE_Original": sum_sq_raw / total_count,
        "MAE_Original": sum_abs_raw / total_count
    }


# 3. Build non-shuffled evaluation loader
def make_eval_loader(dataset):
    return DataLoader(
        dataset,
        batch_size=256,
        shuffle=False,
        drop_last=False,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda"
    )


# 4. Save cumulative metrics from finished runs
def rebuild_metrics_csv():

    rows = []

    for horizon in PRED_LENGTHS:
        for seed in EVAL_SEEDS:

            path = MODEL_DIR / (
                f"linear_h{horizon}_seed{seed}.pt"
            )

            if not path.exists():
                continue

            artifact = torch.load(
                path,
                map_location="cpu",
                weights_only=True
            )

            for split, scores in artifact["scores"].items():
                rows.append({
                    "Dataset": "ETTh1",
                    "Model": "Linear",
                    "Horizon": horizon,
                    "Seed": seed,
                    "Split": split,
                    **scores,
                    "Best_Epoch": artifact["best_epoch"],
                    "Epochs_Trained": artifact["epochs_trained"],
                    "Train_Seconds": artifact["train_seconds"]
                })

    df = pd.DataFrame(rows)

    if len(df):
        df = df.sort_values(
            ["Horizon", "Seed", "Split"]
        ).reset_index(drop=True)

        assert not df.duplicated(
            ["Horizon", "Seed", "Split"]
        ).any()

        temp = METRICS_PATH.with_suffix(".tmp")
        df.to_csv(temp, index=False)
        temp.replace(METRICS_PATH)

    return df


# 5. Training function for one horizon and seed
def train_one_linear(horizon, seed):

    run_name = f"linear_h{horizon}_seed{seed}"

    model_path = MODEL_DIR / f"{run_name}.pt"
    latest_path = CHECKPOINT_DIR / f"{run_name}_latest.pt"
    history_path = HISTORY_DIR / f"{run_name}.csv"

    config = {
        "dataset": "ETTh1",
        "model": "Linear",
        "horizon": int(horizon),
        "seed": int(seed),
        "seq_len": SEQ_LEN,
        "channels": N_CHANNELS,
        "individual": INDIVIDUAL,
        "source_sha256": raw_sha256,
        "training": copy.deepcopy(TRAIN_CONFIG)
    }

    # Skip fully completed runs
    if model_path.exists():
        artifact = torch.load(
            model_path,
            map_location="cpu",
            weights_only=True
        )

        assert artifact["config"] == config, (
            f"Config mismatch: {run_name}"
        )

        print(f"SKIP: {run_name} already completed")
        return

    # Independent initialization per seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    datasets, loaders = make_dataloaders(
        horizon=horizon,
        seed=seed
    )

    model = LTSFLinear(
        seq_len=SEQ_LEN,
        pred_len=horizon,
        channels=N_CHANNELS,
        individual=INDIVIDUAL
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=TRAIN_CONFIG["learning_rate"]
    )

    criterion = torch.nn.MSELoss()

    start_epoch = 1
    best_val = float("inf")
    best_epoch = 0
    best_weights = None
    bad_epochs = 0
    history = []
    previous_seconds = 0.0

    # Restore interrupted training
    if latest_path.exists():

        state = torch.load(
            latest_path,
            map_location="cpu",
            weights_only=True
        )

        assert state["config"] == config, (
            f"Checkpoint config mismatch: {run_name}"
        )

        model.load_state_dict(state["model_state_dict"])
        optimizer.load_state_dict(
            state["optimizer_state_dict"]
        )

        # Transfer optimizer tensors to active device
        for opt_state in optimizer.state.values():
            for k, v in opt_state.items():
                if torch.is_tensor(v):
                    opt_state[k] = v.to(DEVICE)

        start_epoch = int(state["epoch"]) + 1
        best_val = float(state["best_val"])
        best_epoch = int(state["best_epoch"])
        best_weights = state["best_weights"]
        bad_epochs = int(state["bad_epochs"])
        history = state["history"]
        previous_seconds = float(
            state["elapsed_seconds"]
        )

        # Restore exact Train shuffle state
        loaders["train"].generator.set_state(
            state["loader_rng"]
        )

        torch.set_rng_state(state["torch_rng"])

        if DEVICE.type == "cuda" and "cuda_rng" in state:
            torch.cuda.set_rng_state_all(
                state["cuda_rng"]
            )

        print(
            f"RESUME: {run_name} "
            f"from epoch {start_epoch - 1}"
        )

    else:
        print(f"START: {run_name}")

    clock = time.perf_counter()
    completed_epochs = start_epoch - 1

    # 6. Epoch training loop
    for epoch in range(
        start_epoch,
        TRAIN_CONFIG["epochs"] + 1
    ):

        model.train()

        train_sum_sq = 0.0
        train_count = 0

        for X, Y in loaders["train"]:

            X = X.to(DEVICE, non_blocking=True)
            Y = Y.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            prediction = model(X)
            loss = criterion(prediction, Y)

            loss.backward()
            optimizer.step()

            train_sum_sq += (
                loss.item() * Y.numel()
            )
            train_count += Y.numel()

        train_mse = train_sum_sq / train_count

        # Evaluate validation with model.eval()
        val_scores = evaluate_linear(
            model, loaders["val"]
        )

        val_mse = val_scores["MSE"]

        # Preserve best validation checkpoint
        if val_mse < best_val:
            best_val = val_mse
            best_epoch = epoch
            bad_epochs = 0

            best_weights = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        else:
            bad_epochs += 1

        history.append({
            "Epoch": epoch,
            "Train_MSE": train_mse,
            "Val_MSE": val_mse,
            "LR": optimizer.param_groups[0]["lr"]
        })

        completed_epochs = epoch

        print(
            f"  Epoch {epoch:2d}/"
            f"{TRAIN_CONFIG['epochs']} | "
            f"Train={train_mse:.6f} | "
            f"Val={val_mse:.6f} | "
            f"Best={best_val:.6f} "
            f"(epoch {best_epoch})"
        )

        # Official type1 LR adjustment occurs
        # at the end of each epoch.
        if TRAIN_CONFIG["lr_schedule"] == "type1":
            next_lr = (
                TRAIN_CONFIG["learning_rate"]
                * (0.5 ** (epoch - 1))
            )

            for group in optimizer.param_groups:
                group["lr"] = next_lr

        # Save latest state every epoch
        elapsed = (
            previous_seconds
            + time.perf_counter() - clock
        )

        progress = {
            "config": config,
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val": best_val,
            "best_epoch": best_epoch,
            "best_weights": best_weights,
            "bad_epochs": bad_epochs,
            "history": history,
            "elapsed_seconds": elapsed,
            "loader_rng": (
                loaders["train"].generator.get_state()
            ),
            "torch_rng": torch.get_rng_state()
        }

        if DEVICE.type == "cuda":
            progress["cuda_rng"] = (
                torch.cuda.get_rng_state_all()
            )

        atomic_save(progress, latest_path)

        if bad_epochs >= TRAIN_CONFIG["patience"]:
            print(
                f"  EARLY STOPPING at epoch {epoch}"
            )
            break

    # 7. Restore best validation weights
    assert best_weights is not None

    model.load_state_dict(best_weights)
    model.eval()

    # Evaluate complete Train / Val / Test
    scores = {}

    for split in ["train", "val", "test"]:
        eval_loader = make_eval_loader(
            datasets[split]
        )

        scores[split] = evaluate_linear(
            model,
            eval_loader
        )

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    elapsed_total = (
        previous_seconds
        + time.perf_counter() - clock
    )

    # 8. Save final complete model artifact
    final_artifact = {
        "config": config,
        "model_state_dict": best_weights,
        "best_epoch": best_epoch,
        "epochs_trained": completed_epochs,
        "best_val_mse": best_val,
        "scores": scores,
        "train_seconds": elapsed_total,
        "history": history
    }

    atomic_save(final_artifact, model_path)

    pd.DataFrame(history).to_csv(
        history_path,
        index=False
    )

    # Final model is saved; intermediate state
    # is no longer needed.
    if latest_path.exists():
        latest_path.unlink()

    # Refresh combined CSV after every run
    rebuild_metrics_csv()

    print(
        f"DONE: {run_name} | "
        f"Best Epoch={best_epoch} | "
        f"Test MSE={scores['test']['MSE']:.6f} | "
        f"MAE={scores['test']['MAE']:.6f} | "
        f"Time={elapsed_total:.1f}s"
    )


# 9. Run the 20 training configurations
TOTAL_RUNS = len(PRED_LENGTHS) * len(EVAL_SEEDS)

print("=== E7 — LINEAR REPRODUCTION TRAINING ===")
print("Device:", DEVICE)
print("Planned runs:", TOTAL_RUNS)
print("Maximum epochs:", TRAIN_CONFIG["epochs"])
print("Checkpointing: EVERY EPOCH")
print()

for horizon in PRED_LENGTHS:
    for seed in EVAL_SEEDS:
        train_one_linear(horizon, seed)


# 10. Final verification
METRICS_DF = rebuild_metrics_csv()

completed = sum(
    (
        MODEL_DIR / f"linear_h{h}_seed{s}.pt"
    ).is_file()
    for h in PRED_LENGTHS
    for s in EVAL_SEEDS
)

assert completed == TOTAL_RUNS
assert len(METRICS_DF) == TOTAL_RUNS * 3

test_df = METRICS_DF[
    METRICS_DF["Split"] == "test"
].copy()

summary_df = (
    test_df.groupby("Horizon", as_index=False)
    .agg(
        MSE_Mean=("MSE", "mean"),
        MSE_Std=("MSE", "std"),
        MAE_Mean=("MAE", "mean"),
        MAE_Std=("MAE", "std")
    )
)

summary_df.to_csv(
    RESULTS_DIR / "metrics_summary.csv",
    index=False
)

print("\n=== FINAL LINEAR RESULTS ===")
display(summary_df.round(6))

print("\nCompleted runs:", completed, "/", TOTAL_RUNS)
print("Metrics rows:", len(METRICS_DF))
print("Summary rows:", len(summary_df))

print("\nLINEAR REPRODUCTION TRAINING PASSED!")


=== E7 — LINEAR REPRODUCTION TRAINING ===
Device: cuda
Planned runs: 20
Maximum epochs: 10
Checkpointing: EVERY EPOCH

START: linear_h96_seed7
  Epoch  1/10 | Train=0.392280 | Val=0.680720 | Best=0.680720 (epoch 1)
  Epoch  2/10 | Train=0.359062 | Val=0.671266 | Best=0.671266 (epoch 2)
  Epoch  3/10 | Train=0.352503 | Val=0.697502 | Best=0.671266 (epoch 2)
  Epoch  4/10 | Train=0.349394 | Val=0.671983 | Best=0.671266 (epoch 2)
  Epoch  5/10 | Train=0.348035 | Val=0.663098 | Best=0.663098 (epoch 5)
  Epoch  6/10 | Train=0.347276 | Val=0.659964 | Best=0.659964 (epoch 6)
  Epoch  7/10 | Train=0.346820 | Val=0.656079 | Best=0.656079 (epoch 7)
  Epoch  8/10 | Train=0.346535 | Val=0.663144 | Best=0.656079 (epoch 7)
  Epoch  9/10 | Train=0.346363 | Val=0.660717 | Best=0.656079 (epoch 7)
  Epoch 10/10 | Train=0.346293 | Val=0.662125 | Best=0.656079 (epoch 7)
  EARLY STOPPING at epoch 10
DONE: linear_h96_seed7 | Best Epoch=7 | Test MSE=0.383510 | MAE=0.395798 | Time=6.6s
START: linear_h96_seed1

,Horizon,MSE_Mean,MSE_Std,MAE_Mean,MAE_Std
0,96,0.384757,0.002373,0.397526,0.003279
1,192,0.441111,0.006991,0.435002,0.007589
2,336,0.486665,0.010111,0.463003,0.010159
3,720,0.525361,0.020417,0.516733,0.015504



Completed runs: 20 / 20
Metrics rows: 60
Summary rows: 4

LINEAR REPRODUCTION TRAINING PASSED!
